# Anima キャラクター LoRA 学習 (Colab)

[kohya-ss/sd-scripts](https://github.com/kohya-ss/sd-scripts) の `anima_train_network.py` で、**Anima base v1.0** 向けのキャラクター LoRA を学習します。

**事前準備**
1. Google Drive に `MyDrive/AnimaLoRA/dataset/` を作成し、学習画像（20枚程度）を入れる
2. キャプション（同名の `.txt`）が無ければ、Step 3 の WD14 タグ付けで自動生成できる
3. ランタイムは GPU（T4 以上）。`bf16` 非対応の GPU（T4）では動作が遅い／不安定な可能性あり

**学習の流れ**: Step 1 環境構築 → Step 2 モデル取得 → Step 3 データセット準備 → Step 4 学習 → Step 5 保存・ComfyUI 連携

> LoRA は DiT のみ学習（`--network_train_unet_only`）します。この形式は ComfyUI でそのまま使えます（Qwen3 LoRA を含める場合のみ変換スクリプトが必要）。
> 学習は base v1.0 に対して行ってください。preview 版で作った LoRA は final で使えない可能性があります。

## Step 1: 環境構築
Google Drive のマウント、sd-scripts のクローン、依存パッケージのインストールを行います。

In [ ]:
#@title 環境構築（Drive マウント / sd-scripts）

import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/AnimaLoRA"   #@param {type:"string"}
DATASET_DIR = f"{PROJECT_DIR}/dataset"
OUTPUT_DIR  = f"{PROJECT_DIR}/output"
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

SD_SCRIPTS = "/content/sd-scripts"
if not os.path.isdir(SD_SCRIPTS):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/kohya-ss/sd-scripts", SD_SCRIPTS], check=True)
else:
    subprocess.run(["git", "-C", SD_SCRIPTS, "pull"], check=True)

%cd /content/sd-scripts
!pip install -q -r requirements.txt
!pip install -q onnx onnxruntime

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
print(f"dataset: {DATASET_DIR}")
print(f"output : {OUTPUT_DIR}")
print("✅ Step 1 完了")

## Step 2: モデルのダウンロード
学習に使うファイルは、ComfyUI 用と同じ Anima 公式リポジトリのものです（学習専用の保存先 `/content/anima_models` に置きます）。

In [ ]:
#@title Anima モデルのダウンロード（学習用）

from huggingface_hub import hf_hub_download
import os, shutil

REPO_ID = "circlestone-labs/Anima"
MODEL_DIR = "/content/anima_models"
os.makedirs(MODEL_DIR, exist_ok=True)

FILES = {
    "dit":  "split_files/diffusion_models/anima-base-v1.0.safetensors",
    "qwen3": "split_files/text_encoders/qwen_3_06b_base.safetensors",
    "vae":  "split_files/vae/qwen_image_vae.safetensors",
}
PATHS = {}
for key, hf_name in FILES.items():
    dest = os.path.join(MODEL_DIR, os.path.basename(hf_name))
    if not os.path.exists(dest):
        print("downloading:", os.path.basename(hf_name))
        tmp = hf_hub_download(repo_id=REPO_ID, filename=hf_name, local_dir="/content/_hf_tmp")
        shutil.move(tmp, dest)
        shutil.rmtree("/content/_hf_tmp", ignore_errors=True)
    PATHS[key] = dest
    print("ok:", dest)
print("✅ Step 2 完了")

## Step 3: データセット準備
- 画像と同名の `.txt` がある → そのまま使う
- 無い → `RUN_WD14=True` で WD14 タガーが自動生成（初回は Hugging Face からタガーモデルを取得）
- **トリガーワード**を各キャプションの先頭に追加します（学習後、生成時のプロンプトにこの語を入れるとキャラが出ます）

> 自動タグは必ず目視で確認してください。**キャラ固有の特徴（髪色・目の色・髪型・固定の服装など）はタグから外す**と、トリガーワード側に特徴が吸収されて再現性が上がります。

In [ ]:
#@title データセット準備（WD14 タグ付け + トリガーワード）

import glob, os, subprocess

TRIGGER_WORD = "mychara"   #@param {type:"string"}  既存タグと被らない造語
RUN_WD14 = True            #@param {type:"boolean"}
WD14_REPO = "SmilingWolf/wd-swinv2-tagger-v3"  #@param {type:"string"}
EXTS = (".png", ".jpg", ".jpeg", ".webp")

images = sorted(p for p in glob.glob(f"{DATASET_DIR}/*") if p.lower().endswith(EXTS))
print(f"画像: {len(images)} 枚")
assert images, f"{DATASET_DIR} に画像を入れてください"

if RUN_WD14:
    missing = [p for p in images if not os.path.exists(os.path.splitext(p)[0] + ".txt")]
    if missing:
        print(f"キャプション無し {len(missing)} 枚 → WD14 でタグ付けします")
        subprocess.run([
            "python", f"{SD_SCRIPTS}/finetune/tag_images_by_wd14_tagger.py",
            "--onnx", "--repo_id", WD14_REPO, "--batch_size", "4",
            "--remove_underscore", "--caption_extension", ".txt",
            DATASET_DIR,
        ], check=True, cwd=SD_SCRIPTS)
    else:
        print("全画像にキャプションがあるためタグ付けをスキップ")

# トリガーワードを先頭に追加（重複追加しない）
for p in images:
    t = os.path.splitext(p)[0] + ".txt"
    text = open(t, encoding="utf-8").read().strip() if os.path.exists(t) else ""
    tags = [x.strip() for x in text.split(",") if x.strip()]
    if not tags or tags[0] != TRIGGER_WORD:
        tags = [TRIGGER_WORD] + [x for x in tags if x != TRIGGER_WORD]
    open(t, "w", encoding="utf-8").write(", ".join(tags))

print("\n--- キャプション例 ---")
for p in images[:3]:
    print(os.path.basename(p), "->", open(os.path.splitext(p)[0] + ".txt", encoding="utf-8").read())
print("✅ Step 3 完了（Drive 上の .txt を直接編集して調整できます）")

## Step 4: 学習（途中再開に対応）
20枚 × `NUM_REPEATS` × `EPOCHS` が総ステップ数です（例: 21枚 × 5 × 10 ≒ 1050 ステップ）。

**途中再開のしくみ**
- 学習結果と学習状態（optimizer 等）は、`NUM_REPEATS` に関係なく **Google Drive** の `AnimaLoRA/output/<名前>_run/` に `SAVE_EVERY_N_STEPS` ごとに保存されます
- Colab の制限で切れたら、**新しいセッションで Step 1 → 2 → 3 → 4 を再実行**するだけで、最新の保存地点から自動で再開します（`RESUME=True`）
- 再開は、データセットと学習設定を**変えずに**行ってください（枚数・`NUM_REPEATS`・`EPOCHS` を変えると、ステップ数が合わなくなります）
- 保存間隔ぶん（既定 100 ステップ）は、切れると再計算になります

**速度の目安**: T4 では bf16 が非対応のため遅くなります（ログで 20 秒/ステップ程度）。短縮したいときは `RESOLUTION` を 768 に下げる、`NUM_REPEATS` / `EPOCHS` を減らす方法があります。

- 過学習ぎみ（背景やポーズが固定される）→ ステップを減らす／途中の保存ファイルを選ぶ
- 学習不足（キャラが安定しない）→ ステップを増やす
- VRAM 不足 → `BLOCKS_TO_SWAP` を 8〜16 に設定（遅くなります）
- loss が NaN → PyTorch 2.5 以上か確認。`MIXED_PRECISION` の切り替えも検討

In [ ]:
#@title 学習設定と実行（途中再開対応）

import subprocess, sys, os, glob

OUTPUT_NAME     = "mychara_anima"   #@param {type:"string"}
RESOLUTION      = 1024              #@param {type:"integer"}
NUM_REPEATS     = 5                 #@param {type:"integer"}
EPOCHS          = 10                #@param {type:"integer"}
SAVE_EVERY_N_STEPS = 100            #@param {type:"integer"}
NETWORK_DIM     = 16                #@param {type:"integer"}
LEARNING_RATE   = "1e-4"            #@param {type:"string"}
MIXED_PRECISION = "bf16"            #@param ["bf16", "fp16"]
BLOCKS_TO_SWAP  = 0                 #@param {type:"integer"}
RESUME          = True              #@param {type:"boolean"}

# 保存先は Drive 上（セッションが切れても残る）
RUN_DIR = f"{OUTPUT_DIR}/{OUTPUT_NAME}_run"
os.makedirs(RUN_DIR, exist_ok=True)

# ── データセット設定 (TOML) ──
toml_path = "/content/anima_dataset.toml"
open(toml_path, "w", encoding="utf-8").write(f"""[general]
# --cache_text_encoder_outputs とは shuffle_caption を併用できない（sd-scripts の制約）
shuffle_caption = false
caption_extension = ".txt"

[[datasets]]
resolution = {RESOLUTION}
batch_size = 1
enable_bucket = true

  [[datasets.subsets]]
  image_dir = "{DATASET_DIR}"
  num_repeats = {NUM_REPEATS}
""")

cmd = [
    "accelerate", "launch", "--num_cpu_threads_per_process", "1",
    "anima_train_network.py",
    f"--pretrained_model_name_or_path={PATHS['dit']}",
    f"--qwen3={PATHS['qwen3']}",
    f"--vae={PATHS['vae']}",
    f"--dataset_config={toml_path}",
    f"--output_dir={RUN_DIR}",
    f"--output_name={OUTPUT_NAME}",
    "--save_model_as=safetensors",
    "--network_module=networks.lora_anima",
    f"--network_dim={NETWORK_DIM}",
    "--network_train_unet_only",
    f"--learning_rate={LEARNING_RATE}",
    "--optimizer_type=AdamW8bit",
    "--lr_scheduler=constant",
    "--timestep_sampling=sigmoid",
    f"--max_train_epochs={EPOCHS}",
    f"--save_every_n_steps={SAVE_EVERY_N_STEPS}",
    "--save_state",
    "--save_last_n_steps_state=300",   # 古い state は削除（Drive 容量節約）。LoRA 本体は全て残る
    "--save_state_on_train_end",
    f"--mixed_precision={MIXED_PRECISION}",
    "--gradient_checkpointing",
    "--cache_latents",
    "--cache_text_encoder_outputs",
    "--vae_chunk_size=64",
    "--vae_disable_cache",
    "--seed=42",
]
if BLOCKS_TO_SWAP > 0:
    cmd.append(f"--blocks_to_swap={BLOCKS_TO_SWAP}")

# ── 途中再開: 最新の state フォルダを探す ──
if RESUME:
    states = sorted(glob.glob(f"{RUN_DIR}/*-state"), key=os.path.getmtime)
    if states:
        print(f"🔄 再開します: {states[-1]}")
        cmd.append(f"--resume={states[-1]}")
    else:
        print("保存済みの state が無いため、最初から学習します")

print(" ".join(cmd), "\n")
proc = subprocess.Popen(cmd, cwd=SD_SCRIPTS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print("\nexit code:", proc.returncode)
assert proc.returncode == 0, "学習に失敗しました。上のログを確認してください"
print("✅ Step 4 完了")

## Step 5: LoRA の選択と保存
`AnimaLoRA/output/<名前>_run/` に、`SAVE_EVERY_N_STEPS` ごとの LoRA（`<名前>-step00000100.safetensors` など）と最終版（`<名前>.safetensors`）が保存されています。
このセルで使うファイルを選び、`AnimaLoRA/output/` 直下にコピーします。ComfyUI 用ノートブックの「LoRA 読み込み」セルは、このフォルダから `models/loras` へコピーします。

> 途中のステップのファイルも試せます。過学習ぎみのときは、少し前のステップのほうが使いやすい場合があります。

In [ ]:
#@title LoRA を選んで Google Drive の output 直下へコピー

import glob, shutil, os

PICK = ""   #@param {type:"string"}  空欄=最終版。例: mychara_anima-step00000500.safetensors

run_files = sorted(glob.glob(f"{RUN_DIR}/*.safetensors"))
print("保存されている LoRA:")
for f in run_files:
    print(" ", os.path.basename(f))
assert run_files, "LoRA ファイルが見つかりません"

name = PICK or f"{OUTPUT_NAME}.safetensors"
src_path = f"{RUN_DIR}/{name}"
assert os.path.exists(src_path), f"{name} が見つかりません（学習が未完了なら PICK に途中のファイルを指定）"

shutil.copy(src_path, OUTPUT_DIR)
print("saved:", os.path.join(OUTPUT_DIR, name))

# 同じランタイムで ComfyUI を動かしている場合は loras フォルダにも置く
comfy_loras = "/content/ComfyUI/models/loras"
if os.path.isdir("/content/ComfyUI"):
    os.makedirs(comfy_loras, exist_ok=True)
    shutil.copy(src_path, comfy_loras)
    print("ComfyUI loras にもコピーしました:", comfy_loras)
print("✅ Step 5 完了")